# Lezione 60 — Demo: il Memory AI Lab al lavoro

> **Modulo:** capstone · **Tempo stimato:** 25 minuti
> **Prerequisiti:** Lezione 58 (pipeline).
>
> Obiettivo pratico unico: eseguire il lab completo su piu' memorie d'esempio,
> osservare i record prodotti e **chiudere il corso**. Il Memory AI Lab funziona
> end-to-end.

## Il lab al lavoro

Istanziamo `MemoryAILab` e lo eseguiamo su un flusso di memorie eterogenee
(episodiche, semantiche, di preferenza). Per ognuna otteniamo il record
strutturato completo — il risultato di 60 lezioni.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

proc = Path('..') / 'datasets' / 'processed'
TIPI = ["episodic", "semantic", "preference"]
_idx = {t: i for i, t in enumerate(TIPI)}
_train = pd.read_csv(proc / 'memory_train.csv')
_train = _train[_train['type'].isin(TIPI)].reset_index(drop=True)

def _tok(t):
    return re.findall(r"[a-zA-Z]+", str(t).lower())

# --- classificatore bag-of-words + softmax (Lezione 54) ---
_vocab = {}
for _t in _train['text']:
    for _w in _tok(_t):
        _vocab.setdefault(_w, len(_vocab))

def _bow(testo):
    v = np.zeros(len(_vocab))
    for w in _tok(testo):
        if w in _vocab:
            v[_vocab[w]] += 1.0
    return v

def _softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)

_Xtr = np.vstack([_bow(t) for t in _train['text']])
_ytr = np.array([_idx[t] for t in _train['type']])
_W = np.zeros((len(_vocab), len(TIPI)))
_b = np.zeros(len(TIPI))
_Y = np.eye(len(TIPI))[_ytr]
for _ in range(300):
    _P = _softmax(_Xtr @ _W + _b)
    _W -= 0.5 * (_Xtr.T @ (_P - _Y) / len(_Xtr) + 1e-3 * _W)
    _b -= 0.5 * (_P - _Y).mean(axis=0)

def classifica_tipo(testo):
    return TIPI[int(_softmax(_bow(testo)[None, :] @ _W + _b).argmax())]

# --- rappresentazione: embedding, entita' (Lezione 55) ---
DIM = 48
def embed(testo):
    v = np.zeros(DIM)
    for w in _tok(testo):
        v[int.from_bytes(w.encode(), 'little') % DIM] += 1.0
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def estrai_entita(testo):
    return [w for w in re.findall(r"\b[A-Z][a-zA-Z]+\b", str(testo)) if w not in {"The", "A", "User"}]

# --- relazioni: fallback a regole (Lezione 56) ---
_VERBI = {"visited": "visited", "met": "met", "likes": "likes",
          "prefers": "prefers", "works": "works_at", "lives": "lives_in"}
def estrai_relazioni(testo):
    ent = estrai_entita(testo)
    for v_sup, v_norm in _VERBI.items():
        if v_sup in testo.lower() and len(ent) >= 2:
            return [{"source": ent[0], "type": v_norm, "target": ent[1]}]
    return []

# --- importanza composita (Lezione 25, versione compatta) ---
_FORTI = {"prefers", "likes", "dislikes", "hates", "loves", "always", "never", "important"}
def calcola_importanza(testo):
    # composita (Lezione 25): lunghezza + parole forti + numero di entita'
    parole = _tok(testo)
    forti = sum(1 for w in parole if w in _FORTI)
    lung = min(len(parole) / 15.0, 1.0)
    n_ent = len(estrai_entita(testo))
    imp = 0.30 * lung + 0.40 * min(forti, 1) + 0.20 * min(n_ent, 2) / 2 + 0.10
    return round(float(min(imp, 1.0)), 2)

In [2]:
import json

class MemoryAILab:
    def __init__(self, soglia_store=0.4):
        self.soglia = soglia_store
        self.archivio = []

    def process(self, memory_id, testo, timestamp=None):
        imp = calcola_importanza(testo)
        rec = {"memory_id": memory_id, "text": testo, "timestamp": timestamp,
               "type": classifica_tipo(testo), "entities": estrai_entita(testo),
               "importance": imp, "should_store": imp >= self.soglia,
               "embedding_dim": DIM, "relations": estrai_relazioni(testo)}
        # archivia solo cio' che vale la pena ricordare, senza duplicare l'id
        gia_presente = any(r["memory_id"] == memory_id for r in self.archivio)
        if rec["should_store"] and not gia_presente:
            self.archivio.append(rec)
        return rec

lab = MemoryAILab()

memorie = [
    ("mem_001", "Marco visited Glasgow with his son.", "2026-07-03"),
    ("mem_002", "The user prefers morning sessions.", "2026-07-04"),
    ("mem_003", "Water boils at 100 degrees.", "2026-07-05"),
    ("mem_004", "ok.", "2026-07-06"),
]
for mid, testo, ts in memorie:
    rec = lab.process(mid, testo, ts)
    tipo, store, imp, ent = rec["type"], rec["should_store"], rec["importance"], rec["entities"]
    print(f"[{tipo:10}] store={store!s:5} imp={imp:.2f}  ent={ent}  {testo}")

[episodic  ] store=True  imp=0.42  ent=['Marco', 'Glasgow']  Marco visited Glasgow with his son.
[preference] store=True  imp=0.60  ent=[]  The user prefers morning sessions.
[semantic  ] store=False imp=0.28  ent=['Water']  Water boils at 100 degrees.
[semantic  ] store=False imp=0.12  ent=[]  ok.


In [3]:
# un record completo, formato JSON (lo schema obiettivo della spec)
print(json.dumps(lab.process("mem_001", "Marco visited Glasgow with his son.",
                             "2026-07-03"), indent=2, ensure_ascii=False))

{
  "memory_id": "mem_001",
  "text": "Marco visited Glasgow with his son.",
  "timestamp": "2026-07-03",
  "type": "episodic",
  "entities": [
    "Marco",
    "Glasgow"
  ],
  "importance": 0.42,
  "should_store": true,
  "embedding_dim": 48,
  "relations": [
    {
      "source": "Marco",
      "type": "visited",
      "target": "Glasgow"
    }
  ]
}


In [4]:
# controlli finali: il lab archivia solo cio' che vale la pena ricordare
assert len(lab.archivio) >= 1
assert all(r["should_store"] for r in lab.archivio)
# la memoria priva di contenuto ("ok.") non viene archiviata
assert not any(r["text"] == "ok." for r in lab.archivio)
print(f"\nmemorie processate: {len(memorie)} | archiviate: {len(lab.archivio)}")
print("Memory AI Lab: funzionante end-to-end.  Corso completato.")


memorie processate: 4 | archiviate: 2
Memory AI Lab: funzionante end-to-end.  Corso completato.


## Riepilogo — il corso in una riga

Da un CSV grezzo di memorie (Fase 1) a un **Memory AI Lab** che pulisce,
classifica, rappresenta, collega, valuta l'importanza, adatta modelli open
(LoRA), impara dalle preferenze e si monitora dal drift. Ogni componente e' stato
costruito **da zero e verificato**, in un percorso di 60 lezioni da 15-30 minuti.

## Quiz finale

1. Quali fasi contribuiscono a un singolo record del lab?
2. Perche' `should_store` evita di archiviare `"ok."`?
3. Cosa faresti quando il monitor (Lezione 59) segnala drift?

*(Risposte: 1. dati/pulizia (1), classificazione (2-3), rappresentazione e grafo
(3-4), Transformer/LoRA (5-6), preference training (7); 2. la sua importanza
composita e' sotto la soglia di memorizzazione; 3. raccoglierei nuovo feedback e
ri-addestrerei/ri-tarerei i componenti, richiudendo il ciclo.)*